In [3]:
"""Export NBA player-by-player, game-by-game box scores from Eoin Moore's Kaggle dataset.

Source: https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores
Input: PlayerStatistics.csv (NOT the season-average CSVs)
Scope: NBA regular-season games from 2018-19 through 2025-26 inclusive.
Output: 8 CSVs (one per season), 1 combined CSV, 1 ZIP, 1 summary README.

Runs in a Kaggle notebook after adding the dataset as an input, or locally:
    python extract_nba_player_game_logs_2018_2026.py --input PlayerStatistics.csv

Each CSV row is ONE player appearance in ONE game, not a season average.
A player who changes teams stays identified by their NBA PlayerID.
No NBA or ESPN APIs are called by the extraction code.
"""
from __future__ import annotations

import argparse
import csv
import json
import re
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

DATASET_URL = "https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores"
SEASON_WINDOWS = {
    "2018-19": ("2018-10-01", "2019-07-01"),
    "2019-20": ("2019-10-01", "2020-11-01"),  # delayed COVID restart
    "2020-21": ("2020-12-01", "2021-08-01"),  # delayed COVID season
    "2021-22": ("2021-10-01", "2022-07-01"),
    "2022-23": ("2022-10-01", "2023-07-01"),
    "2023-24": ("2023-10-01", "2024-07-01"),
    "2024-25": ("2024-10-01", "2025-07-01"),
    "2025-26": ("2025-10-01", "2026-07-01"),
}

# These names are supported by the dataset's PlayerStatistics.csv schema.
COLUMN_MAPPING = {
    "PTS": "points", "REB": "reboundsTotal", "AST": "assists",
    "STL": "steals", "BLK": "blocks", "TOV": "turnovers",
    "PF": "foulsPersonal", "FGM": "fieldGoalsMade", "FGA": "fieldGoalsAttempted",
    "3PM": "threePointersMade", "3PA": "threePointersAttempted",
    "FTM": "freeThrowsMade", "FTA": "freeThrowsAttempted",
    "OREB": "reboundsOffensive", "DREB": "reboundsDefensive",
    "PLUS_MINUS": "plusMinusPoints",
}

OUTPUT_COLUMNS = [
    "PlayerID", "Player", "Season", "GameNumber", "GameID", "Date",
    "Team", "Opponent", "Home", "Win", "MIN",
    "PTS", "REB", "AST", "STL", "BLK", "TOV", "PF",
    "FGM", "FGA", "FG_PCT", "3PM", "3PA", "3P_PCT",
    "FTM", "FTA", "FT_PCT", "OREB", "DREB", "PLUS_MINUS", "TS_PCT",
]
REQUIRED = {"personId", "gameId", "gameDate", "gameType", "firstName", "lastName", "numMinutes"}


def find_source(path=None):
    if path:
        p = Path(path).expanduser()
        if not p.is_file():
            raise FileNotFoundError(f"No file found at {p}")
        return p
    for folder in [Path("/kaggle/input"), Path.cwd(), Path("/mnt/data")]:
        if folder.is_dir():
            found = [p for p in folder.rglob("*") if p.is_file() and p.name.lower() == "playerstatistics.csv"]
            if found:
                return sorted(found)[0]
    raise FileNotFoundError(
        "PlayerStatistics.csv not found. On Kaggle: choose Add Input and attach "
        "eoinamoore/historical-nba-data-and-player-box-scores. "
        "Or supply --input /path/to/PlayerStatistics.csv. "
        "Season-average CSVs cannot be turned back into real game logs."
    )


def get_season(raw_date):
    # Typical Kaggle values: '2024-10-22' or '2024-10-22T00:00:00'.
    match = re.match(r"^\s*(\d{4}-\d{2}-\d{2})", str(raw_date or ""))
    if not match:
        return None, None
    day = match.group(1)
    for season, (start, end) in SEASON_WINDOWS.items():
        if start <= day < end:
            return season, day
    return None, day


def minutes_to_number(raw):
    t = str(raw or "").strip()
    if not t or t.lower() in {"nan", "none", "null"}:
        return None
    if ":" in t:
        chunks = t.split(":")
        try:
            if len(chunks) == 2:
                return int(chunks[0]) + float(chunks[1]) / 60
            if len(chunks) == 3:
                return int(chunks[0]) * 60 + int(chunks[1]) + float(chunks[2]) / 60
        except ValueError:
            return None
    try:
        return float(t)
    except ValueError:
        return None


def safe_number(value):
    v = str(value if value is not None else "").strip()
    if not v or v.lower() in {"nan", "none", "null", "-"}:
        return ""
    try:
        x = float(v)
    except (ValueError, OverflowError):
        return ""
    if not (-1e20 < x < 1e20):
        return ""
    return int(x) if x.is_integer() else round(x, 4)


def ratio_percent(made, attempted):
    try:
        d = float(attempted)
        return round(100 * float(made) / d, 2) if d > 0 and made != "" else ""
    except (ValueError, TypeError, ZeroDivisionError):
        return ""


def full_team(row, name_col, city_col):
    city = str(row.get(city_col, "") or "").strip()
    name = str(row.get(name_col, "") or "").strip()
    return " ".join(x for x in (city, name) if x)


def normalize_row(raw, season, day, minutes):
    player = " ".join(str(raw.get(k, "") or "").strip() for k in ("firstName", "lastName")).strip()
    out = {
        "PlayerID": str(raw.get("personId", "") or "").strip(),
        "Player": player,
        "Season": season,
        "GameNumber": "",  # added after per-player chronological sorting
        "GameID": str(raw.get("gameId", "") or "").strip(),
        "Date": day,
        "Team": full_team(raw, "playerteamName", "playerteamCity"),
        "Opponent": full_team(raw, "opponentteamName", "opponentteamCity"),
        "Home": str(raw.get("home", "") or "").strip(),
        "Win": str(raw.get("win", "") or "").strip(),
        "MIN": round(minutes, 2),
    }
    for output_name, source_name in COLUMN_MAPPING.items():
        out[output_name] = safe_number(raw.get(source_name))
    out["FG_PCT"] = ratio_percent(out["FGM"], out["FGA"])
    out["3P_PCT"] = ratio_percent(out["3PM"], out["3PA"])
    out["FT_PCT"] = ratio_percent(out["FTM"], out["FTA"])
    try:
        pts, fga, fta = float(out["PTS"]), float(out["FGA"]), float(out["FTA"])
        denom = 2 * (fga + 0.44 * fta)
        out["TS_PCT"] = round(100 * pts / denom, 2) if denom else ""
    except (ValueError, TypeError):
        out["TS_PCT"] = ""
    return out


def extract(source, output_dir):
    source = Path(source)
    output_dir = Path(output_dir)
    season_folder = output_dir / "seasons"
    season_folder.mkdir(parents=True, exist_ok=True)
    seen = set()
    records = {season: [] for season in SEASON_WINDOWS}
    counts = Counter()
    game_types = Counter()

    print(f"Reading original game-level Kaggle data: {source}")
    with source.open("r", newline="", encoding="utf-8-sig", errors="replace") as handle:
        reader = csv.DictReader(handle)
        available = set(reader.fieldnames or [])
        missing = REQUIRED - available
        if missing:
            raise ValueError(f"Missing required columns in PlayerStatistics.csv: {sorted(missing)}. "
                             f"Found: {reader.fieldnames}")
        missing_stats = {name for name in COLUMN_MAPPING.values() if name not in available}
        if missing_stats:
            print(f"Note: optional statistics unavailable in source: {sorted(missing_stats)}")

        for raw in reader:
            counts["all_rows"] += 1
            game_type = str(raw.get("gameType", "") or "").strip()
            game_types[game_type] += 1
            # Keep only games officially labelled regular season by the source.
            if re.sub(r"[\s_\-]+", "", game_type.lower()) != "regularseason":
                continue
            season, day = get_season(raw.get("gameDate"))
            if not season:
                continue
            minutes = minutes_to_number(raw.get("numMinutes"))
            if minutes is None or minutes <= 0:  # DNP / inactive / did not dress
                counts["excluded_dnp"] += 1
                continue
            player_id = str(raw.get("personId", "") or "").strip()
            game_id = str(raw.get("gameId", "") or "").strip()
            if not player_id or not game_id:
                counts["excluded_missing_id"] += 1
                continue
            key = (season, player_id, game_id)
            if key in seen:
                counts["excluded_duplicate"] += 1
                continue
            seen.add(key)
            records[season].append(normalize_row(raw, season, day, minutes))
            counts[season] += 1

            if counts["all_rows"] % 300000 == 0:
                print(f"  scanned {counts['all_rows']:,} source rows...")

    if not seen:
        raise ValueError(
            "No regular-season game records matched the requested years. "
            f"Observed gameType values: {dict(game_types.most_common(15))}"
        )

    combined_path = output_dir / "nba_player_game_logs_2018_19_to_2025_26_combined.csv"
    season_paths = []
    with combined_path.open("w", encoding="utf-8", newline="") as master_handle:
        master_writer = csv.DictWriter(master_handle, fieldnames=OUTPUT_COLUMNS)
        master_writer.writeheader()

        for season, game_rows in records.items():
            # Sorting by player/date makes every individual player's game log sequential.
            game_rows.sort(key=lambda r: (r["Player"].casefold(), r["PlayerID"], r["Date"], r["GameID"]))
            player_counters = defaultdict(int)
            for row in game_rows:
                player_counters[row["PlayerID"]] += 1
                row["GameNumber"] = player_counters[row["PlayerID"]]
            path = season_folder / f"nba_player_game_logs_{season.replace('-', '_')}.csv"
            with path.open("w", encoding="utf-8", newline="") as output_handle:
                writer = csv.DictWriter(output_handle, fieldnames=OUTPUT_COLUMNS)
                writer.writeheader()
                writer.writerows(game_rows)
            master_writer.writerows(game_rows)
            season_paths.append(path)
            print(f"{season}: {len(game_rows):,} player-games from {len(player_counters):,} players -> {path.name}")

    summary = {
        "source_url": DATASET_URL,
        "source_file": source.name,
        "season_scope": list(SEASON_WINDOWS),
        "filter": "gameType == Regular Season; MIN > 0; one row per (Season, PlayerID, GameID)",
        "total_source_rows_scanned": counts["all_rows"],
        "total_output_player_games": len(seen),
        "by_season": {season: counts[season] for season in SEASON_WINDOWS},
        "excluded_dnp": counts["excluded_dnp"],
        "excluded_duplicates": counts["excluded_duplicate"],
        "excluded_missing_id": counts["excluded_missing_id"],
        "observed_game_types": dict(game_types),
        "columns": OUTPUT_COLUMNS,
        "notes": [
            "One row is a player in an individual game, NOT a season average.",
            "No playoffs, play-in, preseason, or non-regular-season cup finals.",
            "Percentages are recomputed from game makes and attempts; values are 0-100.",
            "GameNumber is sequential within each player-season.",
            "Source dataset must be checked for licensing before public redistribution.",
        ],
    }
    summary_path = output_dir / "EXTRACTION_SUMMARY.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    archive = output_dir / "nba_player_game_logs_2018_19_to_2025_26.zip"
    with ZipFile(archive, "w", ZIP_DEFLATED, compresslevel=6) as z:
        for path in [*season_paths, combined_path, summary_path]:
            z.write(path, path.relative_to(output_dir))
    print(f"\nDone. {len(seen):,} individual player-game rows; saved ZIP: {archive}")
    print("Data source permissions should be checked before redistributing exported datasets.")
    return archive


def main(argv=None):
    parser = argparse.ArgumentParser(description="Extract eight years of player-game logs from Kaggle PlayerStatistics.csv")
    parser.add_argument("--input", type=str, default=None, help="Path to PlayerStatistics.csv; auto-detected on Kaggle")
    parser.add_argument("--output-dir", type=str, default=None)
    args = parser.parse_args(argv)
    source = find_source(args.input)
    out = args.output_dir or ("/kaggle/working/nba_player_game_logs" if Path("/kaggle/working").is_dir() else "nba_player_game_logs")
    return extract(source, out)




In [4]:
# Runs the extraction and creates a downloadable ZIP.
from IPython.display import display, FileLink
archive_path = main([])
display(FileLink(str(archive_path)))


Reading original game-level Kaggle data: /kaggle/input/datasets/eoinamoore/historical-nba-data-and-player-box-scores/PlayerStatistics.csv
2018-19: 25,895 player-games from 526 players -> nba_player_game_logs_2018_19.csv
2019-20: 22,215 player-games from 528 players -> nba_player_game_logs_2019_20.csv
2020-21: 22,841 player-games from 540 players -> nba_player_game_logs_2020_21.csv
2021-22: 25,826 player-games from 602 players -> nba_player_game_logs_2021_22.csv
2022-23: 25,687 player-games from 539 players -> nba_player_game_logs_2022_23.csv
2023-24: 24,715 player-games from 570 players -> nba_player_game_logs_2023_24.csv
2024-25: 26,203 player-games from 569 players -> nba_player_game_logs_2024_25.csv
2025-26: 26,648 player-games from 582 players -> nba_player_game_logs_2025_26.csv

Done. 200,030 individual player-game rows; saved ZIP: /kaggle/working/nba_player_game_logs/nba_player_game_logs_2018_19_to_2025_26.zip
Data source permissions should be checked before redistributing export

/kaggle/working/nba_player_game_logs/nba_player_game_logs_2018_19_to_2025_26.zip